# DexYCB validation of the hand Kalman filters

Measures how accurate the filters are against **real ground truth**: 30 fps RGB video with per-frame 3D hand joints ([DexYCB](https://dex-ycb.github.io/), subject 01, one camera).

Six methods are scored on the same frames:

| method | angles in | filter |
|---|---|---|
| `mp` | MediaPipe | none |
| `corr` | v12 corrector | none |
| `kf_angle_mp` | MediaPipe | angle-only KF |
| `kf_angle_corr` | v12 corrector | angle-only KF |
| `ekf_mp` | MediaPipe | EKF, angles + MediaPipe 3D |
| `ekf_corr` | v12 corrector | EKF, angles + MediaPipe 3D |

**Before you start**
1. Upload the whole `kalman_filtering` folder to `MyDrive/kalman_filtering` (you can leave out `output/`).
2. A CPU runtime is enough.
3. The first run downloads 12.4 GB into Colab's temporary disk (not your Drive), keeps one camera, and saves that subset (under ~1 GB) to your Drive. Later runs reuse the subset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Settings

In [ ]:
KF_DRIVE = '/content/drive/MyDrive/kalman_filtering'        # where you uploaded the folder
SUBSET_TAR = '/content/drive/MyDrive/dexycb_s01_cam836212060125.tar'   # cached subset (made on first run)
SHORTCUT = '/content/drive/MyDrive/20200709-subject-01.tar.gz'         # optional "Add shortcut" fallback
RESULTS_DRIVE = '/content/drive/MyDrive/kalman_filtering_results/dexycb'
MP_CACHE = RESULTS_DRIVE + '/mp_cache'   # MediaPipe detections, kept across sessions

SUBJECT_ID = '1Ehh92wDE3CWAiKG7E9E73HjN2Xk2XfEk'   # 20200709-subject-01.tar.gz, 12.4 GB
CALIB_ID = '1UAwVKT4Rgb1fLcFoa1o71_-0NtSvvLAQ'     # calibration.tar.gz, 14 KB
SERIAL = '836212060125'                           # camera to use (any of the 8)

KF = '/content/kalman_filtering'   # local working copy of the code
DATA = '/content/dexycb'           # extracted images + labels
OUT = '/content/results/dexycb'    # results, copied to Drive at the end

## 2 · Code and packages

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd):
    """Run a shell command and stream its output into the cell."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'inference_feedback' in line or line.startswith(('W0000', 'I0000')):
            continue
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'command failed ({p.returncode}): {cmd}')

assert os.path.isdir(KF_DRIVE), f'upload the kalman_filtering folder to {KF_DRIVE} first'
if os.path.exists(KF):
    shutil.rmtree(KF)
shutil.copytree(KF_DRIVE, KF, ignore=shutil.ignore_patterns('output', '__pycache__', '*.mp4'))
os.makedirs(MP_CACHE, exist_ok=True)

sh('pip -q install mediapipe gdown pyyaml')
sh('python -c "import mediapipe, torch, numpy; print(mediapipe.__version__, torch.__version__, numpy.__version__)"')

Run the self-test. It must end with `ALL CHECKS PASSED`.

In [ ]:
sh(f'cd {KF} && python tests/selftest.py 2>&1 | tail -n 30')

## 3 · Data

- **First time:** downloads subject 01, extracts only camera `SERIAL` (colour images, labels, `meta.yml`) and saves that subset to your Drive. The extraction reads the whole 12 GB archive, so allow roughly 10–20 min in total.
- **Later:** restores the subset from Drive in about a minute.
- **If `gdown` fails** with "too many users have viewed or downloaded this file": open the [file](https://drive.google.com/file/d/1Ehh92wDE3CWAiKG7E9E73HjN2Xk2XfEk), choose ⋮ → Organize → **Add shortcut** → My Drive, then re-run this cell. The shortcut path in `SHORTCUT` is used instead.

In [ ]:
os.makedirs(DATA, exist_ok=True)
if os.path.exists(SUBSET_TAR):
    sh(f'tar -xf "{SUBSET_TAR}" -C {DATA}')
else:
    if os.path.exists(SHORTCUT):
        sh(f'cp "{SHORTCUT}" /content/subject01.tar.gz')
    else:
        sh(f'gdown {SUBJECT_ID} -O /content/subject01.tar.gz')
    sh(f'gdown {CALIB_ID} -O /content/calibration.tar.gz')
    sh('ls -lh /content/subject01.tar.gz /content/calibration.tar.gz')
    sh(f"tar -xzf /content/subject01.tar.gz -C {DATA} --wildcards "
       f"'*/{SERIAL}/color_*' '*/{SERIAL}/labels_*' '*/meta.yml'")
    sh(f'tar -xzf /content/calibration.tar.gz -C {DATA}')
    sh(f'tar -cf "{SUBSET_TAR}" -C {DATA} .')
    sh('rm -f /content/subject01.tar.gz')
sh(f'du -sh {DATA}')
sh(f'echo sequences: $(find {DATA} -name meta.yml | wc -l)  images: $(find {DATA} -name "color_*.jpg" | wc -l)')

## 4 · Quick run (5 sequences)

Checks everything works before the full run. MediaPipe detections are cached in `MP_CACHE` on your Drive, so the full run reuses them.

In [ ]:
sh(f'cd {KF} && python validation/validate_dexycb.py --root {DATA} --serial {SERIAL} '
   f'--out {OUT} --cache-dir {MP_CACHE} --max-seqs 5 --tag quick')

## 5 · Full run (all 100 sequences, ~7,000 frames)

MediaPipe on a Colab CPU takes roughly 5–10 min the first time; later runs use the cache.

Useful flags:
- `--sigma-angle 4`, `--jerk-psd 3e4`: filter settings;
- `--calib mediapipe`: build the hand model from MediaPipe instead of the ground truth;
- `--methods mp kf_angle_corr`: run only some methods.

In [ ]:
sh(f'cd {KF} && python validation/validate_dexycb.py --root {DATA} --serial {SERIAL} '
   f'--out {OUT} --cache-dir {MP_CACHE} --tag full')

## 6 · Results

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

RUN = f'{OUT}/full'   # or f'{OUT}/quick'
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
summary = pd.read_csv(f'{RUN}/summary.csv')
display(summary.drop(columns=['description']).round(2))
for png in ('summary.png', 'per_angle_mae.png', 'timeseries_Index_PIP_Flex.png'):
    if os.path.exists(f'{RUN}/{png}'):
        display(Image(filename=f'{RUN}/{png}'))
display(Markdown(open(f'{RUN}/report.md', encoding='utf-8').read()))

### Reading the numbers
- **`angle_mae`** is accuracy against the ground truth. Compare `kf_*` and `ekf_*` with their unfiltered input (`mp` or `corr`): a filter mostly removes **noise**, so expect small MAE gains and large jitter gains.
- **`jitter_vs_gt`**: 1.0 means as smooth as the real motion; well above 1 is jitter; well below 1 means over-smoothing.
- **`accel_err_deg`** is the standard temporal metric for video pose (lower is better).
- **`lag_frames`**: positive means the output trails the ground truth (1 frame = 33 ms).
- **`coast_mae`** is the error on frames the filter bridged while MediaPipe lost the hand.
- **`nis1` / `nis2`**: aim for about 21 / 45.
- **Per-angle bias table in the report:** a filter cannot remove bias; the corrector has to.
- **`pa_mpjpe_mm`**: `mp` is scored on MediaPipe's own landmarks, everything else on FK of its angles. The FK model floor printed in the report is the best any FK method can reach.

## 7 · Tuning sweep (uses cached detections, no MediaPipe)

Grid over the angle-noise σ and the jerk PSD for the angle-only KF on corrected angles. The best setting minimises `angle_mae` or `accel_err_deg` while keeping `jitter_vs_gt` near 1 and `nis1` near 21.

In [ ]:
import itertools, time
import numpy as np
sys.path.insert(0, f'{KF}/core')
sys.path.insert(0, f'{KF}/validation')
import validate_dexycb as V
from pipeline import AngleCorrector

seqs = V.find_sequences(DATA, SERIAL)
prep = V.prepare(seqs, f'{KF}/models/hand_landmarker.task', MP_CACHE,
                 AngleCorrector(f'{KF}/models/angle_corrector_v12.pth'))
base = V.evaluate(prep, V.run_methods(prep, {}, ['corr']))[0].iloc[0]
print(f"unfiltered corrector: MAE {base.angle_mae:.2f}  accel {base.accel_err_deg:.2f}  jitter/GT {base.jitter_vs_gt:.2f}")

rows, t0 = [], time.time()
for sig, q in itertools.product([2, 3, 4, 6, 8], [1e4, 3e4, 1e5, 3e5, 1e6]):
    r = V.evaluate(prep, V.run_methods(prep, dict(sigma_angle=sig, jerk_psd=q), ['kf_angle_corr']))[0].iloc[0]
    rows.append(dict(sigma_angle=sig, jerk_psd=q, angle_mae=r.angle_mae, accel_err=r.accel_err_deg,
                     jitter_vs_gt=r.jitter_vs_gt, lag=r.lag_frames, nis1=r.nis1))
sweep = pd.DataFrame(rows)
print(f'{len(rows)} settings in {time.time() - t0:.0f} s')
display(sweep.pivot(index='sigma_angle', columns='jerk_psd', values='angle_mae').round(2))
display(sweep.pivot(index='sigma_angle', columns='jerk_psd', values='accel_err').round(2))
display(sweep.pivot(index='sigma_angle', columns='jerk_psd', values='nis1').round(1))
display(sweep.sort_values('angle_mae').head(5).round(2))

Optional: tune the EKF's 3D-keypoint noise (slower, about 30 s per setting), using the best angle settings from above.

In [ ]:
best = sweep.sort_values('angle_mae').iloc[0]
rows = []
for sxy, sz in itertools.product([4, 8, 15], [15, 30, 60]):
    cfg = dict(sigma_angle=best.sigma_angle, jerk_psd=best.jerk_psd, sigma_xy=sxy, sigma_z=sz)
    r = V.evaluate(prep, V.run_methods(prep, cfg, ['ekf_corr']))[0].iloc[0]
    rows.append(dict(sigma_xy=sxy, sigma_z=sz, angle_mae=r.angle_mae, accel_err=r.accel_err_deg,
                     pa_mpjpe=r.pa_mpjpe_mm, nis1=r.nis1, nis2=r.nis2))
ekf_sweep = pd.DataFrame(rows)
display(ekf_sweep.round(2))

## 8 · Save results to Drive

In [ ]:
os.makedirs(RESULTS_DRIVE, exist_ok=True)
for name in ('quick', 'full'):
    if os.path.isdir(f'{OUT}/{name}'):
        shutil.copytree(f'{OUT}/{name}', f'{RESULTS_DRIVE}/{name}', dirs_exist_ok=True)
if 'sweep' in globals():
    sweep.to_csv(f'{RESULTS_DRIVE}/sweep_angle_kf.csv', index=False)
if 'ekf_sweep' in globals():
    ekf_sweep.to_csv(f'{RESULTS_DRIVE}/sweep_ekf.csv', index=False)
print('saved to', RESULTS_DRIVE)